# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()

print(f"{getattr(dataset.metadata, 'name', '')}: {getattr(dataset.metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets with their `@id` and associated fields
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet `@id`: {rs['@id']}")
    # List columns (fields) of this record set
    if 'field' in rs:
        print("  Fields/columns:")
        for field in rs['field']:
            # Each field is a dictionary with @id, name, dataType, etc.
            print(f"    - {field['@id']} (name: {field.get('name', 'N/A')}, type: {field.get('dataType', 'N/A')})")
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, we'll demonstrate extracting the main survey data.
# Find the main record set by matching its name or field composition.
main_recordset = None
for rs in record_sets:
    # Heuristically choose the record set with the most fields (likely main survey data)
    if main_recordset is None or ('field' in rs and len(rs['field']) > len(main_recordset.get('field', []))):
        main_recordset = rs

record_sets_ids = [rs['@id'] for rs in record_sets]
print(f"Extracting the following record sets: {record_sets_ids}\n")

dataframes = {}
for record_set in record_sets_ids:
    # Each record is a dict with field @id as keys
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set])} records for record set {record_set}")

# Explore the columns of the main record set
main_rs_id = main_recordset['@id']
print(f"\nColumns for main record set ({main_rs_id}):")
print(list(dataframes[main_rs_id].columns))

# Show the first few records
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming distributions, or grouping data by demographic attributes.

In [ ]:
# Select a numeric field for analysis, e.g., GAD-7 total score (replace with the correct field `@id`)
# We'll auto-detect a likely numeric field based on columns and show how to operate on it
import numpy as np

df = dataframes[main_rs_id]
# Try to guess a GAD-7 field or similar numeric score
numeric_field = None
for col in df.columns:
    # Heuristically pick a column likely to be total_gad7_score, etc.
    if 'gad' in col and ('score' in col or col.endswith('7')):
        numeric_field = col
        break
if numeric_field is None:
    # Fallback: choose the first column with numeric dtype
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

print(f"Analyzing numeric field: {numeric_field}")

# Drop NA to avoid issues
df = df.copy()
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
filtered_df = df[df[numeric_field] > 10]
print(f"Filtered records with {numeric_field} > 10: {len(filtered_df)} records")
print(filtered_df[[numeric_field]].head())

# Normalize the field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a demographic column, e.g., gender or education level
group_field = None
for candidate in ['gender', 'sex', 'level_of_education', 'marital_status', 'village']:
    for col in df.columns:
        if candidate.lower() in col.lower():
            group_field = col
            break
    if group_field:
        break

if group_field:
    print(f"\nGrouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())
else:
    print("No demographic grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

if group_field:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, you've learned to load a FAIR² Croissant-formatted dataset using `mlcroissant`, inspect metadata, and programmatically extract and analyze survey data using record set and field `@id`s. 

We've demonstrated basic numeric field extraction, filtering, normalization, demographic grouping, and simple visualizations. For more advanced or domain-specific analysis, see the Croissant metadata schema and field definitions provided at the dataset source URL. 

You can extend this notebook to perform deeper analysis or to process additional record sets or fields by referencing their full `@id`. This is especially important if you publish derived data or produce FAIR outputs downstream.

For more, visit the [mlcroissant documentation](https://mlcommons.org/croissant/).